### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="homesite_quote_conversion",
    dataset_year="2015",
    domain_str="insurance",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/homesite-quote-conversion",
    download_description="""
We use the train.csv from the Kaggle competition.

kaggle competitions download -c homesite-quote-conversion -f train.csv.zip && unzip train.csv.zip &&  rm train.csv.zip
mkdir -p local-data-warehouse/homesite_quote_conversion && mv train.csv local-data-warehouse/homesite_quote_conversion/
""",
    # References
    academic_reference_bibtex=r"""@misc{Darrel2015HomesiteQuoteConversion,
  author = {Darrel and Stephen D. Stayton and Will Cukierski},
  title  = {Homesite Quote Conversion},
  year   = {2015},
  howpublished = {\url{https://kaggle.com/competitions/homesite-quote-conversion}},
  note   = {Kaggle competition}
}
""",
    academic_reference_bibtex_key="Darrel2015HomesiteQuoteConversion",
    license="Kaggle Competition Rules",
    data_tags=["IID", "Anonymized"],
    curation_comments="""
We start with the train.csv from Kaggle.

- The data has been anonymized, so feature meanings are unknown.
- The data contains decoded spatial information.
- We fix the Field10 column to be integer instead of string with ',' characters.
- We convert the Original_Quote_Date to datetime.
- IMPORTANT NOTE: this dataset was used as a non-IID task by TabRed. We keep it as an IID task, because: (A) the Kaggle discussions regarding validation converged on doing IID cross-validation, (B) based on our understanding of the prediction task IID seems most appropriate, and (C) the official test data on Kaggle contains timestamps from the same period as the train. Concluding from this, we think that treating the data as IID is appropriate and that potential temporal leakage is insignificant (due to A and B) or intended (due to C).
- We drop the ID number and constant columns.
- We replace -1 with NaN (https://www.kaggle.com/competitions/homesite-quote-conversion/discussion/17417) and keep ordinals as numeric features following TabRed's preprocessing.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="QuoteConversion_Flag",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="QuoteConversion_Flag",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "train.csv")
print("Loaded data shape:", df.shape)

df["Field10"] = df["Field10"].astype("string").str.replace(",", "").astype(int)
df["Original_Quote_Date"] = pd.to_datetime(df["Original_Quote_Date"])
as_cat = df.select_dtypes(include=["object"]).columns.tolist() + ["QuoteConversion_Flag"]
df[as_cat] = df[as_cat].astype("category")

ordinal_fields_with_nan_mask = ['CoverageField1A','CoverageField1B','CoverageField11A','CoverageField11B','SalesField2A','SalesField2B','PersonalField4A','PersonalField4B','PersonalField10A','PersonalField10B','PropertyField1A','PropertyField1B','PropertyField2A','PropertyField11A','PropertyField16A','PropertyField16B','PropertyField21A','PropertyField21B','PropertyField24A','PropertyField24B','PropertyField26A','PropertyField26B','PropertyField39A','PropertyField39B','GeographicField1A','GeographicField1B','GeographicField2A','GeographicField2B','GeographicField5A','GeographicField5B','GeographicField6A','GeographicField6B','GeographicField7A','GeographicField7B','GeographicField8A','GeographicField8B','GeographicField9A','GeographicField9B','GeographicField10A','GeographicField10B','GeographicField11A','GeographicField11B','GeographicField12A','GeographicField12B','GeographicField13A','GeographicField13B','GeographicField14A','GeographicField14B','GeographicField15A','GeographicField15B','GeographicField16A','GeographicField16B','GeographicField17A','GeographicField17B','GeographicField18A','GeographicField18B','GeographicField19A','GeographicField19B','GeographicField20A','GeographicField20B','GeographicField21A','GeographicField21B','GeographicField22A','GeographicField22B','GeographicField23A','GeographicField23B','GeographicField24A','GeographicField24B','GeographicField25A','GeographicField25B','GeographicField26A','GeographicField26B','GeographicField27A','GeographicField27B','GeographicField28A','GeographicField28B','GeographicField29A','GeographicField29B','GeographicField30A','GeographicField30B','GeographicField31A','GeographicField31B','GeographicField32A','GeographicField32B','GeographicField33A','GeographicField33B','GeographicField34A','GeographicField34B','GeographicField35A','GeographicField35B','GeographicField36A','GeographicField36B','GeographicField37A','GeographicField37B','GeographicField38A','GeographicField38B','GeographicField39A','GeographicField39B','GeographicField40A','GeographicField40B','GeographicField41A','GeographicField41B','GeographicField42A','GeographicField42B','GeographicField43A','GeographicField43B','GeographicField44A','GeographicField44B','GeographicField45A','GeographicField45B','GeographicField46A','GeographicField46B','GeographicField47A','GeographicField47B','GeographicField48A','GeographicField48B','GeographicField49A','GeographicField49B','GeographicField50A','GeographicField50B','GeographicField51A','GeographicField51B','GeographicField52A','GeographicField52B','GeographicField53A','GeographicField53B','GeographicField54A','GeographicField54B','GeographicField55A','GeographicField55B','GeographicField56A','GeographicField56B','GeographicField57A','GeographicField57B','GeographicField58A','GeographicField58B','GeographicField59A','GeographicField59B','GeographicField60A','GeographicField60B','GeographicField61A','GeographicField61B','GeographicField62A','GeographicField62B']
df[ordinal_fields_with_nan_mask] = df[ordinal_fields_with_nan_mask].replace(-1, np.nan)

df = df.drop(columns=[
    "QuoteNumber", # ID
    # Constant columns
    "GeographicField10A",
    "GeographicField10B"
])
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()